# 09E · Where does the out-of-mask RNA come from?

This notebook tests two questions in the Ada-14R region used for 09B–09D.

1. How much observed RNA lies in the original `tissue-low_transcript-high` squares outside every Proseg mask? Does the myeloid signal track nearby macrophages, and what does the underlying H&E show?
2. When SPARKLE uses only the two H&E-high classes, does it still reduce myeloid signal in pan-T-supported cells while preserving more signal in macrophages?

**Run beside 09D** with the existing **CT26 SPARKLE 0.1.4 (spatial CPU)** kernel. The original pinned **CT26 SPARKLE 0.1.4 (CPU)** kernel also works. Keep the local 09B run, its prepared input, and any completed 09D run folders accessible. The previous return ZIP contains aggregate results. Reuse of a fit requires its local cell-level tables. The notebook locates compatible runs automatically and fits any missing domain in the original tile.

The native evaluation includes only squares marked in-tissue by Space Ranger. The four-class analysis uses the saved Step 00 labels; it also checks how well a raw 0–1 versus ≥2 UMI split reproduces their transcript component. `tissue-low_transcript-high` is a candidate for displaced RNA or weakly stained tissue. Its label alone cannot establish molecular origin.

In [ ]:
from pathlib import Path
from IPython.display import display, Image, Markdown
import json
import pandas as pd
from run09e import run_09e
from review09 import show_review_zip

RESULT_ROOT = Path("09_pilot_results")
RUN_DIR = None  # Optional path to the completed 09B_4403598b0ac3 directory.
STUDY_ROOT = Path("/stash/data/nonclin/TBIO-8110_VisiumHD-Adagrasib-mouseTumor")
TEMP_PROJECT_ROOT = Path("/host_root/nethome/reny28/Projects/Visium_projects/TBIO-8110/tmp")
PATH_OVERRIDES = {}  # Same keys as in 09D, for individually relocated native inputs.
PADDING_UM = 400.
BACKGROUND_GENES = 128
IMAGE_PATH = None  # Optional relocated copy of the original registered Step 00 slide.
CROP_CENTERS_UM = None  # Automatic. Or a list of (Proseg x, Proseg y) in µm in the evaluation core.
CROP_WIDTH_UM = 160.

In [ ]:
out, archive, status = run_09e(
    result_root=RESULT_ROOT, run_dir=RUN_DIR, study_root=STUDY_ROOT, temp_project_root=TEMP_PROJECT_ROOT,
    overrides=PATH_OVERRIDES, padding_um=PADDING_UM, background_genes=BACKGROUND_GENES,
    image_path=IMAGE_PATH, centers_um=CROP_CENTERS_UM, width_um=CROP_WIDTH_UM)
print("Run status:", status["state"], "  |  Local results:", out)
display(pd.DataFrame(status["stages"]).T.fillna(""))
review_download = show_review_zip(archive)
if (out / "readout.md").is_file():
    display(Markdown((out / "readout.md").read_text()))


def read_json(name):
    return json.loads((out / name).read_text())

def show_figure(name, width=960):
    if (out / name).is_file():
        display(Image(filename=str(out / name), width=width))

for stage, result in status["stages"].items():
    if result.get("reason"):
        print(stage, result["reason"])

## 1. Count the four classes directly

These counts cover the buffered region used to fit SPARKLE. The square count includes measured zeros. The percentage columns use all out-of-mask squares or all out-of-mask UMIs as their respective denominators. The heat/stain and transcript labels were saved upstream; squares that disagree with a raw 1.6-UMI cutoff stay in their recorded class. The full CSV also contains the core, the halo alone and assigned squares.

In [ ]:
if (out / "qc_class_coverage.csv").is_file():
    coverage = pd.read_csv(out / "qc_class_coverage.csv")
    bg = coverage.loc[(coverage.scope == "buffered") & (coverage.assignment == "out_of_mask")].copy()
    bg["% of out-of-mask squares"] = 100 * bg.square_fraction_of_assignment
    bg["% of out-of-mask UMIs"] = 100 * bg.UMI_fraction_of_assignment
    display(bg.set_index("qc_class")[["n_squares", "UMIs", "UMIs_per_square", "% of out-of-mask squares",
                                      "% of out-of-mask UMIs", "n_zero", "n_one", "n_two_or_more",
                                      "n_disagree_raw_1p6"]].round(2))
if (out / "ghost_qc.json").is_file():
    audit = read_json("ghost_qc.json")
    print(f"{audit['n_raw_cutoff_disagreements']:,} of {audit['n_buffered_classified']:,} classified "
          "buffered squares disagree with the raw 0–1 vs ≥2 UMI split.")
show_figure("qc_class_coverage.png", 800)

## 2. Is the RNA related to nearby macrophages?

The primary distance plot uses **all out-of-mask tissue-low squares in the core**, pooling the two transcript classes to include genuine zero counts. Compare against the tissue-high squares, then inspect individual classes in `ghost_distance.csv`. The source reference includes previously labeled macrophage/monocyte centroids throughout the prepared section. Distances are center-to-square measurements in micrometres.

Compare raw marker UMIs per square **and** marker UMIs per 1,000 total UMIs. A change in the first can follow a general drop in capture depth. A myeloid-specific decline near-to-far that recurs across local blocks is consistent with regional spillover. A patch following faint tissue on H&E would instead warrant a review of its tissue call. Pan-T and epithelial panels give context for a shared neighborhood gradient. `ghost_within_blocks.csv` compares near (≤25 µm) and farther (>50 through 100 µm) squares within each 200‑µm block. Empty distance groups and sparse counts limit interpretation. One section provides a descriptive spatial test.

In [ ]:
show_figure("ghost_distance.png")
if (out / "ghost_distance.csv").is_file():
    gradient = pd.read_csv(out / "ghost_distance.csv")
    view = gradient.loc[gradient.qc_group.eq("tissue-low_all") & gradient.marker.isin(["Lyz2", "C1qa", "C1qb"])]
    display(view[["marker", "distance_band_um", "n_squares", "total_UMIs", "marker_UMIs",
                  "marker_UMIs_per_square", "marker_per_1000_total_UMIs"]].round(3))
if (out / "ghost_within_blocks.csv").is_file():
    blocks = pd.read_csv(out / "ghost_within_blocks.csv")
    paired_blocks = blocks.loc[(blocks.n_near > 0) & (blocks.n_far > 0)]
    if len(paired_blocks):
        display(paired_blocks.groupby("marker").agg(
            blocks_with_near_and_far=("block_x", "size"),
            blocks_with_depth_in_both=("near_minus_far_per_1000_total_UMIs", "count"),
            median_near_minus_far_UMI_per_square=("near_minus_far_UMIs_per_square", "median"),
            median_near_minus_far_per_1000_UMIs=("near_minus_far_per_1000_total_UMIs", "median")).round(3))
    else:
        print("No eligible block contained both near and farther tissue-low squares.")

## 3. Inspect the same region on H&E

The automatic selection looks for a macrophage-rich crop and a separate lower-density crop with similar tissue coverage. Where available, an additional crop highlights out-of-mask tissue-low/transcript-high squares. It uses prior labels and native square coverage before examining the marker counts. In each crop the same axes show H&E, all four QC classes, out-of-mask candidate squares, and raw *Lyz2*, *C1qa* and *C1qb*. Compare marker-positive squares with visible cell structure, pale tissue and tissue interfaces. The receipt records the image, transformation, crop-selection rule and coordinates so the fields can be reopened.

In [ ]:
if (out / "he_overlay_receipt.json").is_file():
    hist = read_json("he_overlay_receipt.json")
    if hist["state"] == "complete":
        print("Image used:", hist["image"]["kind"], " | ", hist["image"]["path"])
        for region in hist["regions"]:
            show_figure(region["figure"], 1120)
    else:
        print("H&E availability:", hist["state"])
        print(hist.get("recovery", "Inspect he_overlay_receipt.json."))

## 4. Does the H&E-high fit improve the trade-off?

Both fits use the same prespecified lineage genes, model settings and buffered physical region. Each fit selects its abundant background genes and estimates its own decay scale. Removing tissue-low squares can also change the represented cell area, raw counts and availability of background evidence. The comparison is paired on the original core cell IDs represented in **both** domains. Each corrected value is compared with its **own domain's raw hard-mask counts**.

Look for continued myeloid reduction in pan-T-supported cells near macrophages, together with improved preservation of myeloid counts in macrophages and CD3/TCR counts in pan-T-supported cells. Check the number of cells and raw UMIs behind each percentage. If the eligible tissue mask or raw signal changes substantially, inspect `paired_cell_coverage.csv`. The final table reports direct CD4-only, CD8-only, both-positive and neither-positive cells; positive means any count above zero in the recorded `Cd4` or `Cd8a/Cd8b1` panel.

In [ ]:
show_figure("paired_marker_retention.png", 1020)
if (out / "paired_markers.csv").is_file():
    markers = pd.read_csv(out / "paired_markers.csv")
    focus = (markers.group.eq("panT09C") & markers.panel.eq("panT")
           | markers.group.isin(["panT09C_nonmac_near25", "panT09C_nonmac_far50", "macrophage_label"]) & markers.panel.eq("myeloid")
           | markers.group.eq("T_label") & markers.panel.isin(["Cd4", "CD8"])
           | markers.group.eq("B_label") & markers.panel.eq("B"))
    display(markers.loc[focus, ["group","domain","panel","n_shared","raw_panel_UMIs",
                                "corrected_panel_counts","fraction_counts_retained","cells_positive_raw",
                                "cells_positive_corrected"]].round(3))
if (out / "paired_coreceptors.csv").is_file():
    anchors = pd.read_csv(out / "paired_coreceptors.csv")
    display(anchors.loc[anchors.group.eq("T_label")])
if (out / "paired_fits.json").is_file():
    fits = read_json("paired_fits.json")
    display(pd.DataFrame(fits["fits"]).T.reindex(columns=[
        "reused", "n_buffered_squares", "n_matched_core", "n_background_tiles",
        "spatial_decay_um", "n_genes_in_fit", "n_genes_with_reduced_counts"]))
if (out / "background_reconciliation.csv").is_file():
    display(pd.read_csv(out / "background_reconciliation.csv"))

**Return the `09E_return_to_chat_*.zip` linked above.** It contains the aggregate counts, figures including the H&E crops, fit receipts, and any stage-specific diagnostic. Inspect the reported stage status before interpreting missing figures or comparisons. The cell-level fit tables and matrices remain under their respective local 09D run folders.